# GPT-SoVITS v2 - Vietnamese Voice Cloning Pipeline (Optimized)

Notebook này đã được tối ưu hóa để linh hoạt và dễ sử dụng hơn.

### 🚀 Quy trình thực hiện:
1. **Cài đặt môi trường**: Chạy 1 lần đầu tiên.
2. **Cấu hình chung**: Đặt tên model (Experiment Name) tại đây.
3. **Tải Dữ liệu**: Upload file âm thanh từ máy hoặc Drive.
4. **Xử lý Dữ liệu**: Cắt âm thanh và tạo phụ đề (ASR).
5. **WebUI**: Mở giao diện để Train và Inference.

In [12]:
# @title 1. Cài đặt Môi trường & Dependencies (Custom Requirements)
import os
from google.colab import drive
from IPython.display import clear_output

# 1. Mount Drive
if not os.path.exists('/content/drive'):
    try:
        drive.mount('/content/drive')
    except:
        print('Skipping Drive mount')

# 2. Clone Repo
if not os.path.exists("GPT-SoVITS-Vietnamese"):
    print("🚀 Đang clone repository...")
    !git clone https://github.com/tqtuan8788-ai/GPT-SoVITS-Vietnamese.git
    %cd GPT-SoVITS-Vietnamese
else:
    %cd /content/GPT-SoVITS-Vietnamese
    !git pull

# 3. Cài đặt Dependencies Hệ thống
print("🛠️ Đang cài đặt thư viện hệ thống...")
!apt-get update && apt-get install -y libopencc-dev libsndfile1 ffmpeg cmake build-essential

# 4. Cài đặt Python Packages trực tiếp (Không dùng file requirements.txt)
print("📦 Đang cài đặt danh sách thư viện tùy chỉnh...")

# Danh sách các gói bạn yêu cầu
libraries = [
    "numpy<2.0",
    "scipy",
    "tensorboard",
    "librosa==0.10.2",
    "numba",
    "pytorch-lightning>=2.4",
    "gradio<5",
    "ffmpeg-python",
    "tqdm",
    "funasr==1.0.27",
    "cn2an",
    "pypinyin",
    "pyopenjtalk>=0.4.1",
    "g2p_en",
    "torchaudio",
    "modelscope",
    "sentencepiece",
    "transformers>=4.43,<=4.50",
    "peft<0.18.0",
    "chardet",
    "PyYAML",
    "psutil",
    "jieba_fast",
    "jieba",
    "split-lang",
    "fast_langdetect>=0.3.1",
    "wordsegment",
    "rotary_embedding_torch",
    "ToJyutping",
    "g2pk2",
    "ko_pron",
    "python_mecab_ko",
    "fastapi[standard]>=0.115.2",
    "x_transformers",
    "torchmetrics<=1.5",
    "pydantic<=2.10.6",
    "ctranslate2>=4.0,<5",
    "av>=11",
    "onnxruntime-gpu" # Colab dùng x86_64
]

# Cài đặt các gói thông thường
!pip install {" ".join(libraries)}

# Cài đặt opencc với flag --no-binary như bạn yêu cầu
print("🔧 Đang cài đặt OpenCC từ mã nguồn...")
!pip install --no-binary=opencc opencc

# 5. Fix CUDA cho GPU hiệu năng cao (H100/A100)
print("🔥 Đang tối ưu CUDA...")
!pip install -q nvidia-cudnn-cu12 nvidia-cublas-cu12
os.environ['LD_LIBRARY_PATH'] = "/usr/local/lib/python3.10/dist-packages/nvidia/cudnn/lib:/usr/local/lib/python3.10/dist-packages/nvidia/cublas/lib:" + os.environ.get('LD_LIBRARY_PATH', '')

# 6. Tải Pretrained Models
print("📥 Đang tải models...")
base_model_dir = "GPT_SoVITS/pretrained_models"
os.makedirs(base_model_dir, exist_ok=True)
# Tải các file cơ bản
!wget -nc -P {base_model_dir} https://huggingface.co/lj1995/GPT-SoVITS/resolve/main/s2G488k.pth
!wget -nc -P {base_model_dir} https://huggingface.co/lj1995/GPT-SoVITS/resolve/main/s2D488k.pth
!wget -nc -P {base_model_dir} https://huggingface.co/lj1995/GPT-SoVITS/resolve/main/s1bert25Hz-2kh-longer-epoch=68e-step=50232.ckpt
# Cài đặt các thư viện phục vụ cho Whisper và ModelScope
!pip install -q ffmpeg-python modelscope faster-whisper funasr
# Cài đặt OpenCC và các thư viện xử lý ngôn ngữ cần thiết
!pip install opencc-python-reimplemented pypinyin g2p_en
from huggingface_hub import snapshot_download
import os

# Tạo thư mục đích
base_path = "/content/GPT-SoVITS-Vietnamese/GPT_SoVITS/pretrained_models"
os.makedirs(base_path, exist_ok=True)

print("🚚 Đang tải model Pretrained về máy ảo...")

# Tải bộ model GPT-SoVITS gốc
snapshot_download(
    repo_id="lj1995/GPT-SoVITS",
    local_dir=base_path,
    allow_patterns=["*.pth", "*.ckpt", "chinese-roberta-wwm-ext-large/*", "chinese-hubert-base/*"]
)

print("✅ Đã tải xong model nền vào thư mục pretrained_models!")
clear_output()
print("✅ Hoàn tất! Hệ thống đã sẵn sàng với các yêu cầu riêng biệt của bạn.")

🚚 Đang tải model Pretrained về máy ảo...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Fetching 21 files:   0%|          | 0/21 [00:00<?, ?it/s]

✅ Đã tải xong model nền vào thư mục pretrained_models!


In [2]:
# @title ⚙️ 2. Cấu hình Chung (QUAN TRỌNG)
# @markdown Nhập tên experiment của bạn ở đây. Tên này sẽ được dùng xuyên suốt notebook.
exp_name = "Giong_Doc_Sach_01" # @param {type:"string"}

# @markdown Tích chọn nếu muốn tải lại model cũ từ Drive để train tiếp (Resume Training)
resume_from_drive = False # @param {type:"boolean"}

import os
from google.colab import drive

# Common paths
drive_root = "/content/drive/MyDrive/GPT_SoVITS_Models"
dataset_root = "/content/dataset"
output_root = "/content/GPT-SoVITS-Vietnamese/output"

# Mount Drive check
if not os.path.exists('/content/drive'):
    drive.mount('/content/drive')

print(f"✅ Đã cấu hình: {exp_name}")

# Logic Resume
if resume_from_drive:
    drive_exp_path = os.path.join(drive_root, exp_name)
    if os.path.exists(drive_exp_path):
        print(f"🔄 Tìm thấy model cũ trong Drive. Đang khôi phục...")
        # Restore weights
        !cp -r "{drive_exp_path}"/*.pth GPT_SoVITS/weights/ 2>/dev/null || true
        !cp -r "{drive_exp_path}"/*.ckpt GPT_SoVITS/weights/ 2>/dev/null || true
        # Restore logs (checkpoints)
        !mkdir -p logs/{exp_name}
        !cp -r "{drive_exp_path}"/* logs/{exp_name}/ 2>/dev/null || true
        print("✅ Đã khôi phục model thành công!")
    else:
        print(f"⚠️ Không tìm thấy folder {exp_name} trong Drive ({drive_root}). Bỏ qua bước resume.")


✅ Đã cấu hình: Giong_Doc_Sach_01


In [3]:
# @title 3. Tải Dữ liệu Audio
# @markdown Chọn nguồn dữ liệu và upload file. Tự động dùng `exp_name` đã cấu hình ở trên.

import os
from google.colab import files
import shutil

# Use global exp_name
if 'exp_name' not in globals():
    exp_name = "Giong_Doc_Sach_01" # Fallback
    print("⚠️ Warning: exp_name chưa được định nghĩa. Dùng mặc định.")

source_type = "Google Drive" # @param ["Direct Upload", "Google Drive"]
google_drive_path = "/content/drive/MyDrive/my_audio_folder" # @param {type:"string"}

input_audio_folder = f"{dataset_root}/{exp_name}"
os.makedirs(input_audio_folder, exist_ok=True)

if source_type == "Google Drive":
    if os.path.exists(google_drive_path):
        print(f"Dang copy tu {google_drive_path}...")
        for f in os.listdir(google_drive_path):
            if f.endswith((".wav", ".mp3", ".flac")):
                shutil.copy(os.path.join(google_drive_path, f), input_audio_folder)
        print("✅ Da copy xong!")
    else:
        print("❌ Khong tim thay thu muc Google Drive!")

else:
    print("📂 Vui lòng chọn file audio (.wav, .mp3) để upload:")
    uploaded = files.upload()
    for filename in uploaded.keys():
        shutil.move(filename, os.path.join(input_audio_folder, filename))
    print("✅ Upload hoàn tất!")

# Kiểm tra kết quả
if os.path.exists(input_audio_folder):
    files = os.listdir(input_audio_folder)
    print(f"\n📁 Thư mục dataset: {input_audio_folder}")
    print(f"🎵 Số lượng file: {len(files)}")


Dang copy tu /content/drive/MyDrive/my_audio_folder...
✅ Da copy xong!

📁 Thư mục dataset: /content/dataset/Giong_Doc_Sach_01
🎵 Số lượng file: 1


In [6]:
# @title 4. FIX CƯỠNG CHẾ: Cắt Audio & Chạy ASR (GPU + Path Fix)
import os
import sys

# THIẾT LẬP ĐƯỜNG DẪN (QUAN TRỌNG)
%cd /content/GPT-SoVITS-Vietnamese
sys.path.append('/content/GPT-SoVITS-Vietnamese')
os.environ['PYTHONPATH'] = '/content/GPT-SoVITS-Vietnamese'

# 1. Cấu hình
exp_name = "Giong_Doc_Sach_01"
input_folder = f"/content/dataset/{exp_name}"
output_folder = "output/slicer_opt"
os.makedirs(output_folder, exist_ok=True)

# 2. Kiểm tra file .wav
wav_files = [f for f in os.listdir(input_folder) if f.endswith(".wav")]
if not wav_files:
    print("🔄 Đang chuẩn bị file âm thanh...")
    mp3_path = f"{input_folder}/{exp_name}.mp3"
    if os.path.exists(mp3_path):
        !ffmpeg -i "{mp3_path}" -ar 44100 -ac 1 "{input_folder}/{exp_name}.wav" -y -loglevel quiet
        wav_files = [f"{exp_name}.wav"]

if wav_files:
    input_file = os.path.join(input_folder, wav_files[0])

    # 3. CẮT AUDIO (Mỗi đoạn 10 giây)
    print(f"🎬 Đang cắt file: {input_file}")
    !rm -rf {output_folder}/*
    !ffmpeg -i "{input_file}" -f segment -segment_time 10 -c copy "{output_folder}/split_%03d.wav" -loglevel quiet

    # 4. Kiểm tra và chạy ASR với GPU
    sliced_files = os.listdir(output_folder)
    if len(sliced_files) > 0:
        print(f"✅ Đã tạo {len(sliced_files)} đoạn. Bắt đầu chạy ASR bằng GPU...")

        # SỬ DỤNG PYTHONPATH TRỰC TIẾP TRONG LỆNH CHẠY
        !PYTHONPATH=. python tools/asr/fasterwhisper_asr.py -i "{output_folder}" -o "output" -s large-v3 -l vi -p float16

        asr_output_file = f"output/{exp_name}.list"
        generated_list = "output/slicer_opt.list"

        if os.path.exists(generated_list):
            if os.path.exists(asr_output_file): os.remove(asr_output_file)
            os.rename(generated_list, asr_output_file)
            print(f"✅ HOÀN TẤT! File list: {asr_output_file}")
            print("\n--- 5 DÒNG ĐẦU NỘI DUNG ---")
            !head -n 5 {asr_output_file}
        else:
            print("❌ Lỗi: Không xuất được file .list. Có thể do ASR gặp lỗi khi chạy.")
    else:
        print("❌ Không tạo được file cắt.")
else:
    print("❌ Không tìm thấy file đầu vào.")

/content/GPT-SoVITS-Vietnamese
🎬 Đang cắt file: /content/dataset/Giong_Doc_Sach_01/Giong_Doc_Sach_01.wav
✅ Đã tạo 16 đoạn. Bắt đầu chạy ASR bằng GPU...
/usr/local/lib/python3.12/dist-packages/pydub/utils.py:300: SyntaxWarning: invalid escape sequence '\('
  m = re.match('([su]([0-9]{1,2})p?) \(([0-9]{1,2}) bit\)$', token)
/usr/local/lib/python3.12/dist-packages/pydub/utils.py:301: SyntaxWarning: invalid escape sequence '\('
  m2 = re.match('([su]([0-9]{1,2})p?)( \(default\))?$', token)
/usr/local/lib/python3.12/dist-packages/pydub/utils.py:310: SyntaxWarning: invalid escape sequence '\('
  elif re.match('(flt)p?( \(default\))?$', token):
/usr/local/lib/python3.12/dist-packages/pydub/utils.py:314: SyntaxWarning: invalid escape sequence '\('
  elif re.match('(dbl)p?( \(default\))?$', token):
[INFO] Loaded: /usr/local/lib/python3.12/dist-packages/nvidia/cudnn/lib/libcudnn_cnn.so.9
Fetching 5 files:  20% 1/5 [00:00<00:01,  3.19it/s]Warning: You are sending unauthenticated requests to the H

In [ ]:
# @title 5. Khởi động WebUI
# @markdown **Cell này sẽ chạy liên tục.** Làm việc trên link Gradio hiện ra.

import os

%cd /content/GPT-SoVITS-Vietnamese

# Patch config paths
cwd = os.getcwd()
possible_bert_paths = [
    os.path.join(cwd, "GPT_SoVITS/pretrained_models/chinese-roberta-wwm-ext-large"),
    os.path.join(cwd, "pretrained_models/chinese-roberta-wwm-ext-large"),
    "/content/GPT-SoVITS/GPT_SoVITS/pretrained_models/chinese-roberta-wwm-ext-large"
]
found_bert = next((p for p in possible_bert_paths if os.path.exists(p)), possible_bert_paths[0])
found_hubert = found_bert.replace("chinese-roberta-wwm-ext-large", "chinese-hubert-base")

os.environ["cnhubert_base_path"] = found_hubert
os.environ["bert_path"] = found_bert
os.environ["colab_active"] = "1"
os.environ["is_share"] = "True"

print("🚀 Đang khởi động WebUI...")
print("📋 HƯỚNG DẪN:")
print("1. Tab 1 -> 1A: Điền tên experiment và đường dẫn file .list (đã hiện ở bước trên)")
print("2. Bấm Format data")
print("3. Tab 1 -> 1B: Train SoVITS & Train GPT")
print("4. Tab 1 -> 1C: Chọn model và Inference")

!sed -i "s/is_share = False/is_share = True/g" config.py
!python webui.py --share --i18n_dir ./i18n/vi_VN.json

/content/GPT-SoVITS-Vietnamese
🚀 Đang khởi động WebUI...
📋 HƯỚNG DẪN:
1. Tab 1 -> 1A: Điền tên experiment và đường dẫn file .list (đã hiện ở bước trên)
2. Bấm Format data
3. Tab 1 -> 1B: Train SoVITS & Train GPT
4. Tab 1 -> 1C: Chọn model và Inference
* Running on local URL:  http://0.0.0.0:9874
* Running on public URL: https://26f92c2c9e51979219.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
"/usr/bin/python3" -s GPT_SoVITS/prepare_datasets/1-get-text.py
"/usr/bin/python3" -s GPT_SoVITS/prepare_datasets/1-get-text.py
Loading weights: 100% 396/396 [00:00<00:00, 475.76it/s, Materializing param=cls.predictions.transform.dense.weight]
Loading weights: 100% 396/396 [00:00<00:00, 472.18it/s, Materializing param=cls.predictions.transform.dense.weight]
The tied weights mapping and config for this model specifies to tie be

In [ ]:
# @title (Tùy chọn) Chạy lại ASR thủ công
# @markdown Chỉ chạy cell này nếu bạn muốn chạy lại bước tạo phụ đề cho file audio đã cắt.

import os
if 'exp_name' not in globals():
    exp_name = "Giong_Doc_Sach_01"

sliced_audio_folder = "output/slicer_opt"
output_list_file = f"output/{exp_name}.list"

if not os.path.exists(sliced_audio_folder):
    print("❌ LỖI: Không tìm thấy thư mục đã cắt!")
else:
    print("🚀 Đang chạy Whisper (large-v3)...")
    !python tools/asr/fasterwhisper_asr.py -i "{sliced_audio_folder}" -o "output" -s large-v3 -l vi -p float16

    # Rename
    generated = "output/slicer_opt.list"
    if os.path.exists(generated):
        import shutil
        shutil.move(generated, output_list_file)

    print(f"✅ Đã xong: {os.path.abspath(output_list_file)}")

In [ ]:
# @title 6. Lưu Model về Drive
save_to_drive = True # @param {type:"boolean"}

if 'exp_name' not in globals():
    exp_name = "Giong_Doc_Sach_01"

drive_save_path = f"/content/drive/MyDrive/GPT_SoVITS_Models/{exp_name}"

if save_to_drive:
    from google.colab import drive
    if not os.path.exists('/content/drive'):
        drive.mount('/content/drive')

    os.makedirs(drive_save_path, exist_ok=True)
    print(f"Dang luu model {exp_name} vao Drive...")

    !cp GPT_SoVITS/pretrained_models/s2G*.pth "{drive_save_path}"/
    !cp GPT_SoVITS/pretrained_models/s1bert*.ckpt "{drive_save_path}"/
    !cp -r GPT_SoVITS/weights/* "{drive_save_path}"/ 2>/dev/null || true
    !cp -r logs/{exp_name}/* "{drive_save_path}"/ 2>/dev/null || true

    print(f"✅ Da luu xong tai: {drive_save_path}")